In [1]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, f1_score, roc_auc_score
from sklearn.metrics import precision_recall_curve
from medmnist import ChestMNIST
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

Load and prepare data

In [ ]:
train_dataset = ChestMNIST(split='train', download=True, size=64)


# Concatenate images and labels from all splits
X = train_dataset.imgs  # shape (112120, 28, 28)
y = train_dataset.labels


#y[0] = [0 0 0 0 0 0 0 0 0 0 0 0 0 0] #because the first has no disease
#there are 13 possibilities in this dataset. this is way too much so we are going to partially reduce them
#we are going to do this by detecting the 2 most popular diseases, separate them from the group, alongside with the one where it does not have any disease
popular_disease_1 = []
popular_disease_2 = []
no_disease = []

new_y = [] #has to hold the label of pop_diease_1, pop_disease_2 and  "no_disease"

no_disease_idx = np.where(y.sum(axis=1) == 0)[0]
disease_counts = y.sum(axis=0)   # how often each disease appears


d1, d2 = np.argsort(disease_counts)[-2:]   # indices of the two most common diseases
label_names = [train_dataset.info["label"][str(i)] for i in range(len(train_dataset.info["label"]))]
print("Top diseases:", label_names[d1], label_names[d2])

#get indices for each class
d1_idx = np.where(y[:, d1] == 1)[0]
d2_idx = np.where(y[:, d2] == 1)[0]

# Remove overlaps to keep classes cleaner
d1_idx = [i for i in d1_idx if y[i].sum() == 1]
d2_idx = [i for i in d2_idx if y[i].sum() == 1]


#we will truncate them all tho whichever has the shortest len, so that no singular one overwhelms another, then concatinate them, and them shuffle them
min_len = min(len(d1_idx), len(d2_idx), len(no_disease_idx))

#what choice does is it get a ra ramdon min_len amount of samples of the list, and replace= false makes sure it can't get same sample twice
d1_idx = np.random.choice(d1_idx, min_len, replace=False)
d2_idx = np.random.choice(d2_idx, min_len, replace=False)
no_idx = np.random.choice(no_disease_idx, min_len, replace=False)

print(no_idx)
#we build new dataset

indices = np.concatenate([d1_idx, d2_idx, no_idx])

print("indices = " , indices)

new_X = X[indices]
print(f"new_X len = {len(new_X)}")

disease_names = [label_names[d1], label_names[d2], "no_disease"]
num = 0
while num < 3:
    for i in range(min_len):
        new_y.append(disease_names[num])

    num += 1

#flaten from 64 64 to 4096
X_flat = new_X.reshape(new_X.shape[0], -1)
X_flat = X_flat / 255.0  # Normalize pixel values to [0, 1]

print("label names: ", label_names) #label names

unique, counts = np.unique(new_y, return_counts=True)
print("Labels:", unique)
print("Counts:", counts)
print("total len = ", len(new_y))
print("Percentage:", counts / len(new_y) * 100)

print("train dataset shape: ", train_dataset.imgs.shape)
print("precprocessed train dataset shape: ", X_flat.shape)



Top diseases: effusion infiltration
[49693 38783 67670 ... 49259 11882 42820]
indices =  [68054 12744 73945 ... 49259 11882 42820]
new_X len = 8190
label names:  ['atelectasis', 'cardiomegaly', 'effusion', 'infiltration', 'mass', 'nodule', 'pneumonia', 'pneumothorax', 'consolidation', 'edema', 'emphysema', 'fibrosis', 'pleural', 'hernia']
Labels: ['effusion' 'infiltration' 'no_disease']
Counts: [2730 2730 2730]
total len =  8190
Percentage: [33.33333333 33.33333333 33.33333333]
train dataset shape:  (78468, 64, 64)
precprocessed train dataset shape:  (8190, 4096)


split the dataset to test and train

In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    X_flat,
    new_y,
    test_size=0.2, # its a very big dataset, so i can afford to have a big test set, and it will make the training faster.
    #stratify=y, # makes sure the same proportion of each class is in the training and test sets, which is important for imbalanced datasets.
    shuffle=True,
    random_state=42 # make sure the split is reproducible
)

train the random multiforest


Now we predict and evaluate

- these numbers can be misleading, because the dataset is highly imbalanced (most labels are 0), so a model that predicts all zeros would have high accuracy but poor performance on the positive class.
- usually, the higher the f1 score, the better the model is at correctly identifying both positive and negative cases, especially in imbalanced datasets. The AUC values will indicate how well the model can distinguish between the positive and negative classes for each disease.
- for us tho, the f1 micro is at 0, which means that the model is not correctly identifying any of the positive cases, likely due to the class imbalance. The AUC - values will provide more insight into how well the model is performing for each disease, even if the F1 score is low.
 the differnce between micro and macro is that micro calculates metrics globally by counting the total true positives, false negatives and false positives, while macro calculates metrics for each label, and finds their unweighted mean. This does not take label imbalance into account, so it may not be the best choice for our dataset.

In [ ]:
classifier = SVC()

parameters = [{'gamma' : [1, 0.1, 0.01,0.001,0.0001], 'C' : [1,10,100,1000, 10000]}]

grid_search = GridSearchCV(classifier, parameters)

grid_search.fit(x_train, y_train)


more comprehensive prediction

now does the auc check for true positives? i need to make sure its not just writing 00000 everywhere.

In [ ]:
best_estimator = grid_search.best_estimator_


y_pred = best_estimator.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

print("{}% of samples were correctly classified.".format(accuracy * 100))

This is to check if the model only produces 00000 everyhwre, because most images do not have every possible disease so theoretically it could get away with it.

In [ ]:
cm = multilabel_confusion_matrix(y_test, y_pred)

print("True positives per class:", y_test.sum(axis=0))
print("Predicted positives per class:", y_pred.sum(axis=0))

for i, disease in enumerate(label_names):
    tn, fp, fn, tp = cm[i].ravel()
    print(f"{disease}")
    print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
    print()

True positives per class: [219  49 213 337 104 117  27 112  86  45  53  32  57   3]
Predicted positives per class: [0 0 0 2 0 0 0 0 0 0 0 0 0 0]
atelectasis
TP: 0 FP: 0 FN: 219 TN: 1781

cardiomegaly
TP: 0 FP: 0 FN: 49 TN: 1951

effusion
TP: 0 FP: 0 FN: 213 TN: 1787

infiltration
TP: 1 FP: 1 FN: 336 TN: 1662

mass
TP: 0 FP: 0 FN: 104 TN: 1896

nodule
TP: 0 FP: 0 FN: 117 TN: 1883

pneumonia
TP: 0 FP: 0 FN: 27 TN: 1973

pneumothorax
TP: 0 FP: 0 FN: 112 TN: 1888

consolidation
TP: 0 FP: 0 FN: 86 TN: 1914

edema
TP: 0 FP: 0 FN: 45 TN: 1955

emphysema
TP: 0 FP: 0 FN: 53 TN: 1947

fibrosis
TP: 0 FP: 0 FN: 32 TN: 1968

pleural
TP: 0 FP: 0 FN: 57 TN: 1943

hernia
TP: 0 FP: 0 FN: 3 TN: 1997

